# COMP447 Colab First Run

**Goal:** Get ECT running on a Colab T4, verify environment, do a dry-run, and launch the first sanity training.

**Runtime:** Must be **T4 GPU** (Runtime > Change runtime type > T4 GPU in browser Colab, or select T4 when connecting from VS Code Colab extension).

**How to use with AI assistance:** Run each cell, wait for output. If a cell fails, the error is saved into this `.ipynb` file and Claude can read it directly from the repo. No copy-paste needed.

---

## Cell 1: Verify GPU

**Expected output:**
```
torch: 2.x.x+cu...
cuda available: True
gpu: Tesla T4
```

If `cuda available: False` → runtime is not T4. Change runtime type and reconnect.
If gpu is not T4 (e.g. `L4`, `A100`) → still OK for dry run, but final latency numbers must come from T4.

In [1]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram gb:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
vram gb: 102.0


## Cell 2: (skipped) Drive mount — known issue with VS Code extension

**Known issue:** `drive.mount('/content/drive')` fails from the VS Code Colab extension with `MessageError: [dfs_ephemeral] Credentials propagation unsuccessful`. The extension uses its own OAuth flow, incompatible with the inline mount.

**Workaround for today:** skip Drive entirely. Outputs stay in `/content` during the session. For the Monday sanity run this is fine — if the session dies we re-run.

**If you want Drive later:** Command Palette → `Colab: Mount Google Drive to Server...` — this inserts a snippet that uses the extension's own auth and works. Or fall back to browser Colab for runs that need persistence.

The cell below is now a no-op placeholder.

In [2]:
# Drive mount skipped — see markdown above for reason.
# Outputs will live in /content for this session.
import os
os.makedirs("/content/COMP447_runs", exist_ok=True)
print("drive mount skipped, local output dir ready:", "/content/COMP447_runs")

drive mount skipped, local output dir ready: /content/COMP447_runs


## Cell 3: Clone the COMP447 wrapper repo

**Expected output:** listing showing `project`, `final_upload`, `readings`, `proposal_template`, `README.md`.

Note: this is only the wrapper repo. ECT and EDM are cloned fresh in the next cell by `setup_ect.sh`. That's why this clone is fast.

In [3]:
%cd /content
!rm -rf COMP447
!git clone https://github.com/bakaraman/COMP447.git
%cd /content/COMP447
!ls

/content
Cloning into 'COMP447'...
remote: Enumerating objects: 464, done.
remote: Counting objects: 100% (382/382), done.
remote: Compressing objects: 100% (371/371), done.
remote: Total 464 (delta 10), reused 378 (delta 8), pack-reused 82 (from 1)
Receiving objects: 100% (464/464), 131.66 MiB | 84.95 MiB/s, done.
Resolving deltas: 100% (26/26), done.
/content/COMP447
final_upload  project  proposal_template  readings  README.md


## Cell 4: Clone ECT and EDM upstream repos

**Expected output:** two `Cloning into ...` messages, then `Bootstrap complete.` and the "Next steps" block.

This runs `project/scripts/setup_ect.sh` which clones:
- `https://github.com/locuslab/ect.git` into `project/src/ect`
- `https://github.com/NVlabs/edm.git` into `project/src/edm`

In [4]:
%cd /content/COMP447
!bash project/scripts/setup_ect.sh

/content/COMP447
Repo root: /content/COMP447

Cloning or refreshing ECT...
Cloning into '/content/COMP447/project/src/ect'...
remote: Enumerating objects: 123, done.
remote: Counting objects: 100% (123/123), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 123 (delta 65), reused 96 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (123/123), 823.47 KiB | 7.70 MiB/s, done.
Resolving deltas: 100% (65/65), done.

Cloning or refreshing EDM...
Cloning into '/content/COMP447/project/src/edm'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 54 (delta 13), reused 13 (delta 13), pack-reused 20 (from 1)
Receiving objects: 100% (54/54), 2.13 MiB | 14.61 MiB/s, done.
Resolving deltas: 100% (13/13), done.

Bootstrap complete.

Next steps:
  1. Create the runtime environment shown in:
     - /content/COMP447/project/src/ect/env.yml
     - /content/COMP447/project/src/edm

## Cell 5: Install missing Python packages

**Expected output:** a short pip install log, no errors. Colab already has torch, numpy, scipy, pillow, tqdm preinstalled, so this only fills in the rest.

If `diffusers==0.26.3` or `accelerate==0.27.2` pins conflict with Colab's preinstalled versions, we'll relax them — report back.

In [5]:
!pip install -q psutil click requests pyspng imageio-ffmpeg diffusers==0.26.3 accelerate==0.27.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 27.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.9/193.9 kB 25.4 MB/s eta 0:00:00


## Cell 6: Prepare CIFAR-10 in EDM format

**Expected output:**
- `cifar-10-python.tar.gz` downloaded (~170 MB)
- `datasets/cifar10-32x32.zip` created (~180 MB)
- Final `ls -lh datasets` shows the zip

`dataset_tool.py` converts the raw CIFAR to the EDM/ECT training format (zipped uint8 images + labels).

In [6]:
%cd /content/COMP447/project/src/ect
!mkdir -p datasets
!wget -nc -q --show-progress -O /content/cifar-10-python.tar.gz https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
!python3 dataset_tool.py --source=/content/cifar-10-python.tar.gz --dest=datasets/cifar10-32x32.zip
!ls -lh datasets

/content/COMP447/project/src/ect
/content/cifar-10-p 100%[===================>] 162.60M  78.5MB/s    in 2.1s    
  0% 0/50000 [00:00<?, ?it/s]/content/COMP447/project/src/ect/dataset_tool.py:425: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = PIL.Image.fromarray(img, {1: 'L', 3: 'RGB'}[channels])
100% 50000/50000 [00:04<00:00, 10861.88it/s]
total 159M
-rw-r--r-- 1 root root 159M Apr 11 19:21 cifar10-32x32.zip


## Cell 7: Import smoke test

**Expected output:** `psutil ok`, `torch ok`, `cuda: True`, no ImportError.

This verifies that the packages ECT needs at import time are all present in the current kernel.

In [7]:
import sys, importlib
for pkg in ["psutil", "torch", "click", "PIL", "numpy", "scipy", "tqdm", "imageio", "pyspng"]:
    try:
        importlib.import_module(pkg)
        print(f"  ok  {pkg}")
    except ImportError as e:
        print(f"  MISSING  {pkg}: {e}")
import torch
print(f"\npython: {sys.version.split()[0]}")
print(f"torch:  {torch.__version__}")
print(f"cuda:   {torch.cuda.is_available()}")

  ok  psutil
  ok  torch
  ok  click
  ok  PIL
  ok  numpy
  ok  scipy
  ok  tqdm
  ok  imageio
  ok  pyspng

python: 3.12.13
torch:  2.10.0+cu128
cuda:   True


## Cell 8: Dry-run the ECT training command

**Expected output:** ECT parses training options, `dist.init()` initializes with WORLD_SIZE=1, prints the training config JSON, then exits because of `-n` / `--dry_run`. No traceback.

**Why torchrun, not `python3`:** `ct_train.py` calls `dist.init()` which invokes `torch.distributed.init_process_group(init_method='env://')`. That requires MASTER_ADDR, MASTER_PORT, RANK, WORLD_SIZE env vars. `torchrun` sets those automatically for single-process single-node runs. Calling plain `python3` would fail inside `init_process_group`.

**If it still fails, usual suspects:**
- pretrained checkpoint 404 / timeout → retry
- `diffusers==0.26.3` version conflict with torch 2.10 → we'll relax the pin
- NCCL backend errors on single-GPU → unlikely but we'd switch to `gloo`

In [8]:
%cd /content/COMP447/project/src/ect
!torchrun --nnodes=1 --nproc_per_node=1 --rdzv_backend=c10d --rdzv_endpoint=localhost:29501 \
  ct_train.py \
    --outdir=ct-runs-dry \
    --data=datasets/cifar10-32x32.zip \
    --cond=0 \
    --arch=ddpmpp \
    --metrics=fid50k_full \
    --transfer=https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl \
    --duration=0.001 \
    --batch=8 \
    --dry_run

/content/COMP447/project/src/ect

Training options:
{
  "dataset_kwargs": {
    "class_name": "training.dataset.ImageFolderDataset",
    "path": "datasets/cifar10-32x32.zip",
    "use_labels": false,
    "xflip": false,
    "cache": true,
    "resolution": 32,
    "max_size": 50000
  },
  "data_loader_kwargs": {
    "pin_memory": true,
    "num_workers": 1,
    "prefetch_factor": 2
  },
  "network_kwargs": {
    "model_type": "SongUNet",
    "embedding_type": "positional",
    "encoder_type": "standard",
    "decoder_type": "standard",
    "channel_mult_noise": 1,
    "resample_filter": [
      1,
      1
    ],
    "model_channels": 128,
    "channel_mult": [
      2,
      2,
      2
    ],
    "class_name": "training.networks.ECMPrecond",
    "augment_dim": 9,
    "dropout": 0.13,
    "use_fp16": false
  },
  "loss_kwargs": {
    "P_mean": -1.1,
    "P_std": 2.0,
    "q": 2.0,
    "c": 0.0,
    "k": 8.0,
    "b": 1.0,
    "adj": "sigmoid",
    "class_name": "training.loss.ECMLoss"
 

## Cell 8b: Mini training sanity (2-3 minutes)

**Why this cell exists:** Cell 8 (dry run) only validates config parsing and network instantiation. It does **not** run a single training step. This cell runs a few real forward+backward passes so we know:

- pretrained EDM checkpoint actually downloads and loads into the network
- the ECT loss is finite on T4 (not NaN)
- batch 64 fits in 16 GB VRAM (ECT's default batch=128 might OOM)
- torch 2.10 / cuda 12.8 / diffusers 0.26 all agree numerically

**Config:**
- `--duration=0.02` → total 20 kimg (vs Cell 9's 25,600 kimg)
- `--batch=64` → half of ECT default, safe on T4
- `--tick=1` → print a tick line every 1 kimg, so you see ~20 tick logs
- `--eval_every=1000000` + `--dump=1000000` → no FID / snapshot during mini run

**Expected output:** `Loading training set...`, `Constructing network...`, `Setting up EMA...`, then 1-20 `tick N ... kimg X ... loss Y ...` lines with finite loss, then clean exit. Total wall ~2-4 min.

**If the loss is NaN or you see CUDA OOM:** stop and report. We drop batch to 32 or enable mixed precision.

In [9]:
%cd /content/COMP447/project/src/ect

# Patch 1: ECT's InfiniteSampler is incompatible with torch >= 2.2
# torch.utils.data.Sampler.__init__ no longer takes data_source arg.
# Fixes: TypeError: object.__init__() takes exactly one argument
!sed -i 's|super().__init__(dataset)|super().__init__()|' torch_utils/misc.py
!grep -n "super().__init__" torch_utils/misc.py

# NOTE: --metrics=none disables the end-of-training FID50k evaluation which
# takes 5-15 min on T4 and is not needed for a 20-kimg sanity. The 20 ticks
# of training are what we're validating here, not FID quality.
!torchrun --nnodes=1 --nproc_per_node=1 --rdzv_backend=c10d --rdzv_endpoint=localhost:29502 \
  ct_train.py \
    --outdir=ct-runs-mini \
    --data=datasets/cifar10-32x32.zip \
    --cond=0 \
    --arch=ddpmpp \
    --precond=ect \
    --metrics=none \
    --transfer=https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl \
    --duration=0.02 \
    --tick=1 \
    --batch=64 \
    --lr=0.0001 \
    --optim=RAdam \
    --dropout=0.2 \
    --augment=0.0 \
    -q 256 \
    --double=10000 \
    --ema_beta=0.9993 \
    --eval_every=1000000 \
    --dump=1000000 \
    --snap=1000000 \
    --desc=mini_sanity

/content/COMP447/project/src/ect
116:        super().__init__()

Training options:
{
  "dataset_kwargs": {
    "class_name": "training.dataset.ImageFolderDataset",
    "path": "datasets/cifar10-32x32.zip",
    "use_labels": false,
    "xflip": false,
    "cache": true,
    "resolution": 32,
    "max_size": 50000
  },
  "data_loader_kwargs": {
    "pin_memory": true,
    "num_workers": 1,
    "prefetch_factor": 2
  },
  "network_kwargs": {
    "model_type": "SongUNet",
    "embedding_type": "positional",
    "encoder_type": "standard",
    "decoder_type": "standard",
    "channel_mult_noise": 1,
    "resample_filter": [
      1,
      1
    ],
    "model_channels": 128,
    "channel_mult": [
      2,
      2,
      2
    ],
    "class_name": "training.networks.ECMPrecond",
    "dropout": 0.2,
    "use_fp16": false
  },
  "loss_kwargs": {
    "P_mean": -1.1,
    "P_std": 2.0,
    "q": 256.0,
    "c": 0.0,
    "k": 8.0,
    "b": 1.0,
    "adj": "sigmoid",
    "class_name": "training.loss.

## Cell 9: Medium tuning run (OPTIONAL, ~15 min – 1.5 hours depending on GPU)

**DO NOT run `run_ecm_1hour.sh` as-is.** That script's `--duration=25.6` translates to `total_kimg = 25,600`. Measured throughput on this project:

| GPU | sec/kimg | Full schedule (25,600 kimg) |
|---|---|---|
| T4 | ~29 | **~206 hours (~8.6 days)** |
| RTX PRO 6000 Blackwell (measured) | ~2.68 | ~19 hours |
| A100 (estimated) | ~4-5 | ~30-35 hours |
| H100 (estimated) | ~2-3 | ~15-20 hours |

The "1 hour" name comes from the ECT paper's marketing claim, probably measured on faster hardware with multi-GPU scaling. **Not achievable on a single Colab GPU in one session.**

**What we use instead:** a medium-length tuning run at **2,000 kimg** (~7.8% of the full schedule). Enough to get a real (not random-init) ECT checkpoint and a meaningful 2-step FID, not enough to match paper numbers. Good enough to start the latency-matched comparison against Heun.

**Expected wall time by GPU:**

| GPU | Time | FID expectation (2-step) |
|---|---|---|
| Blackwell | ~15 min | probably 30-80 |
| A100 | ~15-20 min | same |
| L4 | ~30 min | same |
| T4 | ~1.5 hours | same |

**Only run this if:**
- You're on Blackwell / H100 / A100 / L4 (not T4 — it's too slow)
- You have compute budget to spare (~5-15 units)
- You plan to evaluate the resulting checkpoint in the next session

**Outputs:**
- Snapshot: `ct-runs-medium/<run>/network-snapshot-002000.pkl`
- 2000 tick lines (with `--tick=10` to reduce output spam)
- Final 2-step FID50k evaluation

In [ ]:
%cd /content/COMP447/project/src/ect

# Patch is idempotent — safe to re-run
!sed -i 's|super().__init__(dataset)|super().__init__()|' torch_utils/misc.py

!torchrun --nnodes=1 --nproc_per_node=1 --rdzv_backend=c10d --rdzv_endpoint=localhost:29503 \
  ct_train.py \
    --outdir=ct-runs-medium \
    --data=datasets/cifar10-32x32.zip \
    --cond=0 \
    --arch=ddpmpp \
    --precond=ect \
    --metrics=fid50k_full \
    --transfer=https://nvlabs-fi-cdn.nvidia.com/edm/pretrained/edm-cifar10-32x32-uncond-vp.pkl \
    --duration=2.0 \
    --tick=10 \
    --batch=128 \
    --lr=0.0001 \
    --optim=RAdam \
    --dropout=0.2 \
    --augment=0.0 \
    -q 256 \
    --double=10000 \
    --ema_beta=0.9993 \
    --eval_every=1000000 \
    --dump=1000000 \
    --snap=1000000 \
    --desc=medium_2k

## Cell 10: (optional) Stash run artifacts locally after Cell 9

Since Drive mount is skipped, this cell just copies ECT run outputs into `/content/COMP447_runs/` so they're in a known place. They'll still vanish when the session ends — download manually via the VS Code file explorer if you want to keep them, or re-run later once Drive mount is set up via the extension command.

In [ ]:
!mkdir -p /content/COMP447_runs/monday_sanity
!cp -r /content/COMP447/project/src/ect/ct-runs/* /content/COMP447_runs/monday_sanity/ 2>/dev/null || echo "nothing to copy yet"
!ls -lh /content/COMP447_runs/